
# Project 1 — fastText + MLP

## Architecture 1: Word Embeddings + Classifier

This notebook implements the first required architecture
for abductive commonsense reasoning on the ART dataset.

Architecture:
- Frozen pretrained fastText embeddings (300d)
- Mean pooling for each of the four text segments
- Concatenation (1200d)
- Linear (1200 → 128)
- ReLU
- Linear (128 → 2)

Only the classifier is trained.


In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn

# Add repository root to Python path
repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load saved preprocessing artifacts
artifact_path = repo_root / "project1" / "artifacts" / "fasttext_art.pt"

artifacts = torch.load(
    artifact_path,
    map_location="cpu",
    weights_only=True
)

vocab = artifacts["vocab"]
embedding_weights = artifacts["embedding_weights"]

# Restore frozen embedding layer
embedding_layer = nn.Embedding.from_pretrained(
    embedding_weights,
    freeze=True,
    padding_idx=vocab["[PAD]"]
)

print("Vocabulary size:", len(vocab))
print("Embedding shape:", embedding_layer.weight.shape)
print("Embeddings frozen:", not embedding_layer.weight.requires_grad)

# Verify that the updated preprocessing artifacts are loaded
assert len(vocab) == 15038, (
    f"Expected updated vocabulary (15038), got {len(vocab)}"
)

assert tuple(embedding_weights.shape) == (15038, 300)
assert not embedding_layer.weight.requires_grad
assert torch.all(embedding_weights[vocab["[PAD]"]] == 0)

print("\nUpdated preprocessing artifacts: PASS")


Vocabulary size: 15038
Embedding shape: torch.Size([15038, 300])
Embeddings frozen: True

Updated preprocessing artifacts: PASS


In [2]:
from datasets import load_dataset
from project1.src.data_loader import create_dataloader

# Load ART dataset
dataset = load_dataset("amarfurt/art-nlp-hs26")

# Create DataLoaders
batch_size = 32

train_loader = create_dataloader(
    dataset["train"],
    vocab,
    batch_size=batch_size,
    shuffle=True
)

val_loader = create_dataloader(
    dataset["validation"],
    vocab,
    batch_size=batch_size,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training batches: 526
Validation batches: 32


In [3]:
# Get one batch
batch = next(iter(train_loader))

input_ids = batch["input_ids"]
lengths = batch["lengths"]
labels = batch["labels"]

# Pass through frozen embedding layer
embeddings = embedding_layer(input_ids)

print("Input IDs:", input_ids.shape)
print("Embeddings:", embeddings.shape)
print("Labels:", labels.shape)

# Verify padding
print("\nSequence lengths correct:",
      torch.equal(
          lengths,
          (input_ids != vocab["[PAD]"]).sum(dim=1)
      ))

# Verify that each example has exactly 3 separators
sep_counts = (input_ids == vocab["[SEP]"]).sum(dim=1)

print("Each example has 3 separators:",
      torch.all(sep_counts == 3).item())


Input IDs: torch.Size([32, 67])
Embeddings: torch.Size([32, 67, 300])
Labels: torch.Size([32])

Sequence lengths correct: True
Each example has 3 separators: True


In [4]:

from project1.src.models import EmbeddingMLP

# Initialize model
model = EmbeddingMLP(
    embedding_layer=embedding_layer,
    pad_id=vocab["[PAD]"],
    sep_id=vocab["[SEP]"],
    hidden_dim=128
)

# Use our existing batch
batch = next(iter(train_loader))

# Forward pass
with torch.no_grad():
    logits = model(batch["input_ids"])

print("Input shape:", batch["input_ids"].shape)
print("Output shape:", logits.shape)

print("\nFirst 5 logits:")
print(logits[:5])

print("\nPredicted classes:")
print(logits.argmax(dim=1)[:5])

print("\nActual classes:")
print(batch["labels"][:5])

# Check trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", trainable_params)
print("Embeddings frozen:", not model.embedding.weight.requires_grad)


Input shape: torch.Size([32, 56])
Output shape: torch.Size([32, 2])

First 5 logits:
tensor([[0.0958, 0.0845],
        [0.0885, 0.0794],
        [0.0922, 0.0823],
        [0.0894, 0.0789],
        [0.0880, 0.0769]])

Predicted classes:
tensor([0, 0, 0, 0, 0])

Actual classes:
tensor([0, 0, 0, 1, 0])

Trainable parameters: 153986
Embeddings frozen: True


In [5]:
#don't run
import wandb

# Initialize a test experiment
run = wandb.init(
    project="nlp-art-hs26-project1",
    name="setup-test",
    job_type="setup",
    config={
        "model": "fasttext-mlp",
        "embedding_dim": 300,
        "hidden_dim": 128,
        "batch_size": 32
    }
)

# Log one test metric
run.log({"test/connection": 1})

print("Project:", run.project)
print("Entity:", run.entity)
print("Run URL:", run.url)

# Finish the experiment
run.finish()


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Project: nlp-art-hs26-project1
Entity: kimbortis-hochschule-luzern
Run URL: https://forge.coreweave.com/wandb/kimbortis-hochschule-luzern/nlp-art-hs26-project1/runs/yl357st7


test/connection,▁
test/connection,1


In [5]:
import torch.nn as nn
from project1.src.training import evaluate

# Our loss function
criterion = nn.CrossEntropyLoss()

# Use CPU for now
device = torch.device("cpu")
model = model.to(device)

# Evaluate the untrained model
val_metrics = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print(f"Validation loss: {val_metrics['loss']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.2%}")


Validation loss: 0.6932
Validation accuracy: 50.50%


In [ ]:
#don't run
import random
import numpy as np
import torch
import torch.nn as nn

from project1.src.models import EmbeddingMLP
from project1.src.training import train_one_epoch, evaluate

# Reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Random seed:", seed)


# Hyperparameters
config = {
    "architecture": "fasttext-mlp",
    "embedding_dim": 300,
    "hidden_dim": 128,
    "batch_size": 32,
    "learning_rate": 0.001,
    "optimizer": "Adam",
    "max_epochs": 60,
    "patience": 15,
    "early_stopping": True,
    "seed": seed
}

# Dynamic device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Initialize a new model
model = EmbeddingMLP(
    embedding_layer=embedding_layer,
    pad_id=vocab["[PAD]"],
    sep_id=vocab["[SEP]"],
    hidden_dim=config["hidden_dim"]
).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config["learning_rate"]
)

print("Device:", device)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", config["learning_rate"])
print("Training examples:", len(train_loader.dataset))
print("Validation examples:", len(val_loader.dataset))


In [6]:
#don't run
import random
import numpy as np
import torch
import torch.nn as nn
import wandb

from pathlib import Path
from project1.src.models import EmbeddingMLP
from project1.src.training import train_one_epoch, evaluate

# Learning rates to compare
learning_rates = [0.0003, 0.001, 0.003]

# Fixed settings for a fair comparison
max_epochs = 60
patience = 15
seed = 42

checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

results = []

for lr in learning_rates:

    print(f"\n{'=' * 55}")
    print(f"EXPERIMENT: Learning rate = {lr}")
    print(f"{'=' * 55}")

    # Reset random seeds for every experiment
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Fresh model with identical initialization
    experiment_model = EmbeddingMLP(
        embedding_layer=embedding_layer,
        pad_id=vocab["[PAD]"],
        sep_id=vocab["[SEP]"],
        hidden_dim=128
    ).to(device)

    # Fresh optimizer
    experiment_optimizer = torch.optim.Adam(
        filter(
            lambda p: p.requires_grad,
            experiment_model.parameters()
        ),
        lr=lr
    )

    experiment_criterion = nn.CrossEntropyLoss()

    run_name = (
        f"mlp_h128_lr{lr}_bs32_seed42_"
        "preprocV2_ES60"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="mlp-learning-rate",
        job_type="hyperparameter-tuning",
        name=run_name,
        config={
            "architecture": "fasttext-mlp",
            "hidden_dim": 128,
            "learning_rate": lr,
            "batch_size": 32,
            "optimizer": "Adam",
            "dropout": 0.0,
            "weight_decay": 0.0,
            "seed": seed,
            "max_epochs": max_epochs,
            "patience": patience,
            "preprocessing": "v2",
            "vocab_size": len(vocab)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    best_accuracy = -1.0
    loss_at_best_accuracy = float("inf")
    lowest_val_loss = float("inf")
    best_epoch = 0
    no_improvement = 0
    last_epoch = 0

    try:
        for epoch in range(1, max_epochs + 1):
            last_epoch = epoch

            train_metrics = train_one_epoch(
                experiment_model,
                train_loader,
                experiment_optimizer,
                experiment_criterion,
                device
            )

            val_metrics = evaluate(
                experiment_model,
                val_loader,
                experiment_criterion,
                device
            )

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            # Track lowest validation loss independently
            lowest_val_loss = min(
                lowest_val_loss,
                val_metrics["loss"]
            )

            # Select checkpoint by validation accuracy
            # Break accuracy ties using validation loss
            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] < loss_at_best_accuracy
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                loss_at_best_accuracy = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )

            else:
                no_improvement += 1

            if epoch == 1 or epoch % 5 == 0 or improved:
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Val: {val_metrics['accuracy']:.2%} | "
                    f"Val loss: {val_metrics['loss']:.4f}"
                )

            if no_improvement >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = (
            loss_at_best_accuracy
        )
        run.summary["lowest_val_loss"] = lowest_val_loss
        run.summary["epochs_trained"] = last_epoch

        results.append({
            "learning_rate": lr,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "loss_at_best": loss_at_best_accuracy,
            "lowest_val_loss": lowest_val_loss,
            "epochs_trained": last_epoch,
            "checkpoint": str(checkpoint_path)
        })

        print(f"\nBest accuracy: {best_accuracy:.2%}")
        print(f"Best epoch: {best_epoch}")
        print(f"Lowest validation loss: {lowest_val_loss:.4f}")

    finally:
        run.finish()

# Compare results
print("\n=== LEARNING RATE COMPARISON ===")

results = sorted(
    results,
    key=lambda x: (
        -x["best_accuracy"],
        x["loss_at_best"]
    )
)

for result in results:
    print(
        f"LR: {result['learning_rate']:<7} | "
        f"Best val acc: {result['best_accuracy']:.2%} | "
        f"Epoch: {result['best_epoch']:02d} | "
        f"Loss at best: {result['loss_at_best']:.4f}"
    )

print("\nBest learning rate:", results[0]["learning_rate"])


EXPERIMENT: Learning rate = 0.0003


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Epoch 01 | Train: 52.68% | Val: 52.80% | Val loss: 0.6881
Epoch 02 | Train: 56.98% | Val: 56.00% | Val loss: 0.6757
Epoch 03 | Train: 60.18% | Val: 58.70% | Val loss: 0.6674
Epoch 04 | Train: 61.59% | Val: 59.30% | Val loss: 0.6593
Epoch 05 | Train: 62.99% | Val: 60.90% | Val loss: 0.6560
Epoch 10 | Train: 66.29% | Val: 60.10% | Val loss: 0.6458
Epoch 12 | Train: 66.98% | Val: 61.00% | Val loss: 0.6506
Epoch 14 | Train: 67.23% | Val: 61.10% | Val loss: 0.6429
Epoch 15 | Train: 67.75% | Val: 60.00% | Val loss: 0.6588
Epoch 18 | Train: 68.70% | Val: 61.50% | Val loss: 0.6517
Epoch 20 | Train: 69.06% | Val: 61.70% | Val loss: 0.6439
Epoch 25 | Train: 70.42% | Val: 60.30% | Val loss: 0.6627
Epoch 27 | Train: 71.34% | Val: 62.30% | Val loss: 0.6510
Epoch 28 | Train: 71.64% | Val: 62.90% | Val loss: 0.6474
Epoch 30 | Train: 72.33% | Val: 62.90% | Val loss: 0.6534
Epoch 35 | Train: 74.10% | Val: 63.00% | Val loss: 0.6512
Epoch 40 | Train: 75.67% | Val: 62.60% | Val loss: 0.6559
Epoch 41 | Tra

epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
train/accuracy,▁▂▂▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████
train/loss,██▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁
val/accuracy,▁▃▅▆▇▆▅▆▆▇▇▆▆▇▆▇▆▇▆▆█▇▆▇▇▇▇███▇▅▇█▇▇▇▅▇▇
val/loss,▆▅▄▃▃▂▂▂▁▂▂▁▃▂▁▂▁▂▄▂▃▂▂▃▃▂▃▅▃▃▅▄▄▅▄▆▆█▆▇
best_epoch,41
best_val_accuracy,0.631
epoch,56
epochs_trained,56
lowest_val_loss,0.64291
train/accuracy,0.81096



EXPERIMENT: Learning rate = 0.001


Epoch 01 | Train: 51.93% | Val: 52.90% | Val loss: 0.6889
Epoch 02 | Train: 57.39% | Val: 56.30% | Val loss: 0.6746
Epoch 03 | Train: 61.16% | Val: 58.70% | Val loss: 0.6649
Epoch 04 | Train: 62.57% | Val: 60.10% | Val loss: 0.6553
Epoch 05 | Train: 63.75% | Val: 59.50% | Val loss: 0.6589
Epoch 06 | Train: 65.17% | Val: 60.50% | Val loss: 0.6496
Epoch 10 | Train: 67.17% | Val: 60.20% | Val loss: 0.6473
Epoch 11 | Train: 67.56% | Val: 60.70% | Val loss: 0.6561
Epoch 12 | Train: 68.04% | Val: 61.70% | Val loss: 0.6596
Epoch 15 | Train: 69.94% | Val: 62.30% | Val loss: 0.6514
Epoch 20 | Train: 72.47% | Val: 62.50% | Val loss: 0.6568
Epoch 25 | Train: 74.47% | Val: 61.70% | Val loss: 0.6659
Epoch 30 | Train: 77.39% | Val: 60.40% | Val loss: 0.6989
Epoch 35 | Train: 80.20% | Val: 60.30% | Val loss: 0.6917
Early stopping at epoch 35

Best accuracy: 62.50%
Best epoch: 20
Lowest validation loss: 0.6439


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/accuracy,▁▂▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████
train/loss,██▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
val/accuracy,▁▃▅▆▆▇▆▆▆▆▇▇▇▇█▇▇█▇█▇▆▅█▇▇▇█▇▆▇▇▇▇▆
val/loss,▆▄▃▂▃▂▂▂▃▁▂▃▂▁▂▂▁▂▂▂▆▇▆▄▄▆▅▅█▇▇▆▇▇▆
best_epoch,20
best_val_accuracy,0.625
epoch,35
epochs_trained,35
lowest_val_loss,0.64392
train/accuracy,0.80198



EXPERIMENT: Learning rate = 0.003


Epoch 01 | Train: 51.79% | Val: 52.90% | Val loss: 0.6878
Epoch 02 | Train: 58.05% | Val: 56.40% | Val loss: 0.6737
Epoch 03 | Train: 61.65% | Val: 59.40% | Val loss: 0.6596
Epoch 04 | Train: 63.37% | Val: 60.50% | Val loss: 0.6511
Epoch 05 | Train: 64.31% | Val: 58.60% | Val loss: 0.6625
Epoch 06 | Train: 65.19% | Val: 61.30% | Val loss: 0.6474
Epoch 07 | Train: 65.80% | Val: 61.90% | Val loss: 0.6448
Epoch 10 | Train: 67.10% | Val: 62.00% | Val loss: 0.6469
Epoch 15 | Train: 68.15% | Val: 59.60% | Val loss: 0.6497
Epoch 20 | Train: 68.75% | Val: 60.50% | Val loss: 0.6493
Epoch 25 | Train: 69.40% | Val: 60.90% | Val loss: 0.6556
Early stopping at epoch 25

Best accuracy: 62.00%
Best epoch: 10
Lowest validation loss: 0.6448


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/accuracy,▁▃▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████
train/loss,█▇▆▅▅▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
val/accuracy,▁▄▆▇▅▇█▇▆█▇█▆▇▆█▇█▇▇█▆▇█▇
val/loss,█▆▃▂▄▁▁▂▃▁▃▂▂▁▂▁▁▃▃▂▅▄▃▁▃
best_epoch,10
best_val_accuracy,0.62
epoch,25
epochs_trained,25
lowest_val_loss,0.64485
train/accuracy,0.69401



=== LEARNING RATE COMPARISON ===
LR: 0.0003  | Best val acc: 63.10% | Epoch: 41 | Loss at best: 0.6598
LR: 0.001   | Best val acc: 62.50% | Epoch: 20 | Loss at best: 0.6568
LR: 0.003   | Best val acc: 62.00% | Epoch: 10 | Loss at best: 0.6469

Best learning rate: 0.0003


In [6]:
#don't run
import importlib
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import wandb

from project1.src import models, training

# Reload updated models.py
importlib.reload(models)
importlib.reload(training)

# Compare without and with dropout
dropout_values = [0.0, 0.2]

learning_rate = 0.0003
max_epochs = 60
patience = 15
seed = 42

checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

dropout_results = []

for dropout in dropout_values:

    print(f"\n{'=' * 55}")
    print(f"DROPOUT EXPERIMENT: p = {dropout}")
    print(f"{'=' * 55}")

    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Fresh frozen embedding layer
    fresh_embedding = nn.Embedding.from_pretrained(
        embedding_weights.detach().clone(),
        freeze=True,
        padding_idx=vocab["[PAD]"]
    )

    # Fresh model
    experiment_model = models.EmbeddingMLP(
        embedding_layer=fresh_embedding,
        pad_id=vocab["[PAD]"],
        sep_id=vocab["[SEP]"],
        hidden_dim=128,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        filter(
            lambda p: p.requires_grad,
            experiment_model.parameters()
        ),
        lr=learning_rate
    )

    criterion = nn.CrossEntropyLoss()

    dropout_name = str(dropout).replace(".", "p")
    run_name = (
        f"mlp_h128_lr{learning_rate}_bs32_"
        f"seed42_drop{dropout_name}_preprocV2"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="mlp-dropout",
        job_type="hyperparameter-tuning",
        name=run_name,
        config={
            "architecture": "fasttext-mlp",
            "hidden_dim": 128,
            "learning_rate": learning_rate,
            "batch_size": 32,
            "dropout": dropout,
            "optimizer": "Adam",
            "weight_decay": 0.0,
            "seed": seed,
            "max_epochs": max_epochs,
            "patience": patience,
            "preprocessing": "v2"
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    best_accuracy = -1.0
    best_loss = float("inf")
    lowest_val_loss = float("inf")
    best_epoch = 0
    no_improvement = 0

    try:
        for epoch in range(1, max_epochs + 1):

            train_metrics = training.train_one_epoch(
                experiment_model,
                train_loader,
                optimizer,
                criterion,
                device
            )

            val_metrics = training.evaluate(
                experiment_model,
                val_loader,
                criterion,
                device
            )

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            lowest_val_loss = min(
                lowest_val_loss,
                val_metrics["loss"]
            )

            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] < best_loss
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                best_loss = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )
            else:
                no_improvement += 1

            if epoch == 1 or epoch % 5 == 0:
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Val: {val_metrics['accuracy']:.2%} | "
                    f"Val loss: {val_metrics['loss']:.4f}"
                )

            if no_improvement >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        # Restore and verify best checkpoint
        experiment_model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=True
            )
        )

        restored = training.evaluate(
            experiment_model,
            val_loader,
            criterion,
            device
        )

        assert abs(
            restored["accuracy"] - best_accuracy
        ) < 1e-6

        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = best_loss
        run.summary["lowest_val_loss"] = lowest_val_loss
        run.summary["epochs_trained"] = epoch

        dropout_results.append({
            "dropout": dropout,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "loss_at_best": best_loss,
            "lowest_val_loss": lowest_val_loss,
            "epochs_trained": epoch
        })

        print(f"\nBest accuracy: {best_accuracy:.2%}")
        print(f"Best epoch: {best_epoch}")
        print(f"Lowest validation loss: {lowest_val_loss:.4f}")
        print("Checkpoint verification: PASS")

    finally:
        run.finish()

# Final comparison
print("\n=== DROPOUT COMPARISON ===")

for result in dropout_results:
    print(
        f"Dropout: {result['dropout']:.1f} | "
        f"Best val acc: {result['best_accuracy']:.2%} | "
        f"Best epoch: {result['best_epoch']} | "
        f"Lowest val loss: {result['lowest_val_loss']:.4f}"
    )



DROPOUT EXPERIMENT: p = 0.0


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Epoch 01 | Train: 52.68% | Val: 52.80% | Val loss: 0.6881
Epoch 05 | Train: 62.99% | Val: 60.90% | Val loss: 0.6560
Epoch 10 | Train: 66.29% | Val: 60.10% | Val loss: 0.6458
Epoch 15 | Train: 67.75% | Val: 60.00% | Val loss: 0.6588
Epoch 20 | Train: 69.06% | Val: 61.70% | Val loss: 0.6439
Epoch 25 | Train: 70.42% | Val: 60.30% | Val loss: 0.6627
Epoch 30 | Train: 72.33% | Val: 62.90% | Val loss: 0.6534
Epoch 35 | Train: 74.10% | Val: 63.00% | Val loss: 0.6512
Epoch 40 | Train: 75.67% | Val: 62.60% | Val loss: 0.6559
Epoch 45 | Train: 77.98% | Val: 62.20% | Val loss: 0.6670
Epoch 50 | Train: 79.51% | Val: 59.10% | Val loss: 0.7310
Epoch 55 | Train: 80.91% | Val: 59.60% | Val loss: 0.7050
Early stopping at epoch 56

Best accuracy: 63.10%
Best epoch: 41
Lowest validation loss: 0.6429
Checkpoint verification: PASS


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/accuracy,▁▂▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████
train/loss,██▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
val/accuracy,▁▃▅▅▇▆▅▆▆▇▇▆▆▆▇▇▆▆▅▇▇▆▇▇▇▇▇███▇▇▇▇▇▇▇▇▇▇
val/loss,▆▄▃▃▂▂▂▂▁▃▁▂▂▁▂▂▃▂▂▁▃▃▃▃▂▃▃▅▃▃▄▄▄▅▄▆▆█▆▇
best_epoch,41
best_val_accuracy,0.631
epoch,56
epochs_trained,56
lowest_val_loss,0.64291
train/accuracy,0.81096



DROPOUT EXPERIMENT: p = 0.2


Epoch 01 | Train: 51.82% | Val: 53.00% | Val loss: 0.6897
Epoch 05 | Train: 61.72% | Val: 58.70% | Val loss: 0.6614
Epoch 10 | Train: 65.72% | Val: 59.70% | Val loss: 0.6574
Epoch 15 | Train: 66.86% | Val: 60.60% | Val loss: 0.6526
Epoch 20 | Train: 68.23% | Val: 60.60% | Val loss: 0.6609
Epoch 25 | Train: 69.68% | Val: 60.10% | Val loss: 0.6511
Epoch 30 | Train: 71.16% | Val: 61.50% | Val loss: 0.6428
Epoch 35 | Train: 72.44% | Val: 62.60% | Val loss: 0.6397
Epoch 40 | Train: 74.41% | Val: 63.00% | Val loss: 0.6477
Epoch 45 | Train: 76.22% | Val: 63.70% | Val loss: 0.6470
Epoch 50 | Train: 77.47% | Val: 63.00% | Val loss: 0.6495
Epoch 55 | Train: 79.57% | Val: 62.30% | Val loss: 0.6628
Epoch 60 | Train: 81.41% | Val: 62.40% | Val loss: 0.6623

Best accuracy: 64.50%
Best epoch: 58
Lowest validation loss: 0.6389
Checkpoint verification: PASS


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/accuracy,▁▂▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████
train/loss,█▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁
val/accuracy,▁▁▄▅▅▄▆▅▅▆▅▆▆▆▅▆▆▆▆▆▇▇▇▇▆▇█▇███▇▇▆▇▇▇▇▆▇
val/loss,█▇▅▄▄▄▂▄▃▂▃▃▂▂▂▂▁▂▂▂▂▂▂▁▁▂▁▁▂▂▂▂▂▂▃▂▃▃▄▅
best_epoch,58
best_val_accuracy,0.645
epoch,60
epochs_trained,60
lowest_val_loss,0.63889
train/accuracy,0.81412



=== DROPOUT COMPARISON ===
Dropout: 0.0 | Best val acc: 63.10% | Best epoch: 41 | Lowest val loss: 0.6429
Dropout: 0.2 | Best val acc: 64.50% | Best epoch: 58 | Lowest val loss: 0.6389


In [6]:
#don't run
import random
import numpy as np
import torch
import torch.nn as nn
import wandb

from project1.src import models, training

# Optimizer / weight decay comparison
experiments = [
    {"optimizer": "Adam", "weight_decay": 0.0},
    {"optimizer": "AdamW", "weight_decay": 0.0},
    {"optimizer": "AdamW", "weight_decay": 0.0001},
]

# Fixed hyperparameters
learning_rate = 0.0003
dropout = 0.2
hidden_dim = 128
max_epochs = 90
patience = 20
seed = 42

checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

weight_decay_results = []

for exp in experiments:

    optimizer_name = exp["optimizer"]
    weight_decay = exp["weight_decay"]

    print(f"\n{'=' * 55}")
    print(f"{optimizer_name} | Weight decay: {weight_decay}")
    print(f"{'=' * 55}")

    # Reset seeds for each experiment
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Fresh frozen embeddings
    fresh_embedding = nn.Embedding.from_pretrained(
        embedding_weights.detach().cpu().clone(),
        freeze=True,
        padding_idx=vocab["[PAD]"]
    )

    # Fresh model
    experiment_model = models.EmbeddingMLP(
        embedding_layer=fresh_embedding,
        pad_id=vocab["[PAD]"],
        sep_id=vocab["[SEP]"],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    parameters = [
        p for p in experiment_model.parameters()
        if p.requires_grad
    ]

    # Select optimizer
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            parameters,
            lr=learning_rate,
            weight_decay=weight_decay
        )
    else:
        optimizer = torch.optim.AdamW(
            parameters,
            lr=learning_rate,
            weight_decay=weight_decay
        )

    criterion = nn.CrossEntropyLoss()

    wd_name = str(weight_decay).replace(".", "p")
    run_name = (
        f"mlp_h128_lr0.0003_bs32_drop0p2_"
        f"{optimizer_name.lower()}_wd{wd_name}_seed42"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="mlp-weight-decay",
        job_type="hyperparameter-tuning",
        name=run_name,
        config={
            "architecture": "fasttext-mlp",
            "hidden_dim": hidden_dim,
            "learning_rate": learning_rate,
            "batch_size": 32,
            "dropout": dropout,
            "optimizer": optimizer_name,
            "weight_decay": weight_decay,
            "seed": seed,
            "max_epochs": max_epochs,
            "patience": patience,
            "preprocessing": "v2"
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    best_accuracy = -1.0
    best_loss = float("inf")
    lowest_val_loss = float("inf")
    best_epoch = 0
    no_improvement = 0

    try:
        for epoch in range(1, max_epochs + 1):

            train_metrics = training.train_one_epoch(
                experiment_model,
                train_loader,
                optimizer,
                criterion,
                device
            )

            val_metrics = training.evaluate(
                experiment_model,
                val_loader,
                criterion,
                device
            )

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            lowest_val_loss = min(
                lowest_val_loss,
                val_metrics["loss"]
            )

            # Best checkpoint selected by validation accuracy
            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] < best_loss
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                best_loss = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )
            else:
                no_improvement += 1

            if epoch == 1 or epoch % 5 == 0:
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Val: {val_metrics['accuracy']:.2%} | "
                    f"Val loss: {val_metrics['loss']:.4f}"
                )

            if no_improvement >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        # Restore best model
        experiment_model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=True
            )
        )

        restored = training.evaluate(
            experiment_model,
            val_loader,
            criterion,
            device
        )

        assert abs(
            restored["accuracy"] - best_accuracy
        ) < 1e-6

        # Save results to W&B
        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = best_loss
        run.summary["lowest_val_loss"] = lowest_val_loss
        run.summary["epochs_trained"] = epoch

        weight_decay_results.append({
            "optimizer": optimizer_name,
            "weight_decay": weight_decay,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "loss_at_best": best_loss,
            "lowest_val_loss": lowest_val_loss,
            "epochs_trained": epoch,
            "checkpoint": str(checkpoint_path)
        })

        print(f"\nBest accuracy: {best_accuracy:.2%}")
        print(f"Best epoch: {best_epoch}")
        print(f"Lowest validation loss: {lowest_val_loss:.4f}")
        print("Checkpoint verification: PASS")

    finally:
        run.finish()

# Final comparison
print("\n=== WEIGHT DECAY COMPARISON ===")

weight_decay_results.sort(
    key=lambda r: (
        -r["best_accuracy"],
        r["loss_at_best"]
    )
)

for result in weight_decay_results:
    print(
        f"{result['optimizer']:5} | "
        f"WD: {result['weight_decay']:<7} | "
        f"Best acc: {result['best_accuracy']:.2%} | "
        f"Epoch: {result['best_epoch']:02d} | "
        f"Lowest loss: {result['lowest_val_loss']:.4f}"
    )

print("\nBest configuration:", weight_decay_results[0])


Adam | Weight decay: 0.0


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Epoch 01 | Train: 51.82% | Val: 53.00% | Val loss: 0.6897
Epoch 05 | Train: 61.72% | Val: 58.70% | Val loss: 0.6614
Epoch 10 | Train: 65.72% | Val: 59.70% | Val loss: 0.6574
Epoch 15 | Train: 66.86% | Val: 60.60% | Val loss: 0.6526
Epoch 20 | Train: 68.23% | Val: 60.60% | Val loss: 0.6609
Epoch 25 | Train: 69.68% | Val: 60.10% | Val loss: 0.6511
Epoch 30 | Train: 71.16% | Val: 61.50% | Val loss: 0.6428
Epoch 35 | Train: 72.44% | Val: 62.60% | Val loss: 0.6397
Epoch 40 | Train: 74.41% | Val: 63.00% | Val loss: 0.6477
Epoch 45 | Train: 76.22% | Val: 63.70% | Val loss: 0.6470
Epoch 50 | Train: 77.47% | Val: 63.00% | Val loss: 0.6495
Epoch 55 | Train: 79.57% | Val: 62.30% | Val loss: 0.6628
Epoch 60 | Train: 81.41% | Val: 62.40% | Val loss: 0.6623
Epoch 65 | Train: 82.80% | Val: 63.30% | Val loss: 0.6737
Epoch 70 | Train: 84.39% | Val: 62.40% | Val loss: 0.6945
Epoch 75 | Train: 85.51% | Val: 63.40% | Val loss: 0.6877
Epoch 80 | Train: 86.99% | Val: 64.00% | Val loss: 0.7079
Epoch 85 | Tra

epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/accuracy,▁▂▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████
train/loss,█▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
val/accuracy,▁▁▄▅▅▅▅▅▅▆▆▅▆▇▆▆▇▇▇▇▇▇▇▇▆▇▆▆▇▇▇█▆▇▇▇▇▇▇▇
val/loss,▃▃▃▂▂▂▁▂▁▃▁▂▂▂▁▁▁▁▁▂▁▂▁▂▂▃▃▃▃▄▄▄▅▅▆▆█▇▇█
best_epoch,69
best_val_accuracy,0.65
epoch,89
epochs_trained,89
lowest_val_loss,0.63889
train/accuracy,0.88935



AdamW | Weight decay: 0.0


Epoch 01 | Train: 51.82% | Val: 53.00% | Val loss: 0.6897
Epoch 05 | Train: 61.72% | Val: 58.70% | Val loss: 0.6614
Epoch 10 | Train: 65.72% | Val: 59.70% | Val loss: 0.6574
Epoch 15 | Train: 66.86% | Val: 60.60% | Val loss: 0.6526
Epoch 20 | Train: 68.23% | Val: 60.60% | Val loss: 0.6609
Epoch 25 | Train: 69.68% | Val: 60.10% | Val loss: 0.6511
Epoch 30 | Train: 71.16% | Val: 61.50% | Val loss: 0.6428
Epoch 35 | Train: 72.44% | Val: 62.60% | Val loss: 0.6397


... (3 outputs left out)


AdamW | Weight decay: 0.0001


Epoch 01 | Train: 51.82% | Val: 53.10% | Val loss: 0.6897
Epoch 05 | Train: 61.76% | Val: 58.70% | Val loss: 0.6615
Epoch 10 | Train: 65.66% | Val: 59.60% | Val loss: 0.6575
Epoch 15 | Train: 66.83% | Val: 60.80% | Val loss: 0.6524
Epoch 20 | Train: 68.04% | Val: 61.30% | Val loss: 0.6629
Epoch 25 | Train: 69.40% | Val: 60.60% | Val loss: 0.6507
Epoch 30 | Train: 70.81% | Val: 62.30% | Val loss: 0.6436
Epoch 35 | Train: 72.38% | Val: 62.60% | Val loss: 0.6410
Epoch 40 | Train: 73.85% | Val: 62.60% | Val loss: 0.6485
Epoch 45 | Train: 75.78% | Val: 63.40% | Val loss: 0.6490
Epoch 50 | Train: 77.32% | Val: 64.30% | Val loss: 0.6497
Epoch 55 | Train: 79.15% | Val: 62.50% | Val loss: 0.6676
Epoch 60 | Train: 81.08% | Val: 62.60% | Val loss: 0.6642
Epoch 65 | Train: 82.48% | Val: 63.10% | Val loss: 0.6728
Epoch 70 | Train: 84.22% | Val: 63.10% | Val loss: 0.6949
Epoch 75 | Train: 85.60% | Val: 63.40% | Val loss: 0.6925
Epoch 80 | Train: 86.95% | Val: 63.80% | Val loss: 0.7057
Early stopping

epoch,▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/accuracy,▁▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
train/loss,███▇▇▇▇▆▆▆▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁
val/accuracy,▁▅▅▆▆▆▆▆▆▆▆▆▆▆▇▆▇▇▇▇▇██▇█▇▇▇▇██▇██▇███▇▇
val/loss,▃▃▂▃▃▂▂▂▃▂▂▂▁▂▂▁▁▂▁▁▂▂▂▂▂▃▄▃▄▃▃▃▄▄▅▆▆▇▇█
best_epoch,63
best_val_accuracy,0.649
epoch,83
epochs_trained,83
lowest_val_loss,0.63968
train/accuracy,0.87566



=== WEIGHT DECAY COMPARISON ===
Adam  | WD: 0.0     | Best acc: 65.00% | Epoch: 69 | Lowest loss: 0.6389
AdamW | WD: 0.0     | Best acc: 65.00% | Epoch: 69 | Lowest loss: 0.6389
AdamW | WD: 0.0001  | Best acc: 64.90% | Epoch: 63 | Lowest loss: 0.6397

Best configuration: {'optimizer': 'Adam', 'weight_decay': 0.0, 'best_accuracy': 0.65, 'best_epoch': 69, 'loss_at_best': 0.6795523262023926, 'lowest_val_loss': 0.6388904962539673, 'epochs_trained': 89, 'checkpoint': 'C:\\Users\\kimbo\\Documents\\HSLU\\Sem 5\\NLP\\nlp-art-hs26-kim\\project1\\artifacts\\mlp_h128_lr0.0003_bs32_drop0p2_adam_wd0p0_seed42_w18vyupc_best.pt'}


In [6]:

import random
import statistics
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import wandb

from project1.src import models, training

# Multi-seed validation
seeds = [42, 123, 2026]
dropouts = [0.0, 0.2]

lr = 0.0003
max_epochs = 90
patience = 20

checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

multiseed_results = []

for seed in seeds:
    for dropout in dropouts:

        # Reset RNGs
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        # Fresh embeddings and classifier
        fresh_embedding = nn.Embedding.from_pretrained(
            embedding_weights.detach().cpu().clone(),
            freeze=True,
            padding_idx=vocab["[PAD]"]
        )

        model_run = models.EmbeddingMLP(
            embedding_layer=fresh_embedding,
            pad_id=vocab["[PAD]"],
            sep_id=vocab["[SEP]"],
            hidden_dim=128,
            dropout=dropout
        ).to(device)

        optimizer_run = torch.optim.Adam(
            [p for p in model_run.parameters()
             if p.requires_grad],
            lr=lr,
            weight_decay=0.0
        )

        criterion_run = nn.CrossEntropyLoss()

        run_name = (
            f"mlp_multiseed_s{seed}_"
            f"drop{str(dropout).replace('.', 'p')}"
            "_lr0p0003"
        )

        run = wandb.init(
            entity="kimbortis-hochschule-luzern",
            project="nlp-art-hs26-project1",
            group="mlp-multiseed",
            name=run_name,
            config={
                "seed": seed,
                "dropout": dropout,
                "learning_rate": lr,
                "hidden_dim": 128,
                "batch_size": 32,
                "optimizer": "Adam",
                "weight_decay": 0.0,
                "max_epochs": max_epochs,
                "patience": patience,
                "preprocessing": "v2"
            }
        )

        run.define_metric("epoch")
        run.define_metric("train/*", step_metric="epoch")
        run.define_metric("val/*", step_metric="epoch")

        best_acc = -1.0
        best_loss = float("inf")
        best_epoch = 0
        no_improvement = 0

        checkpoint = (
            checkpoint_dir /
            f"{run_name}_{run.id}_best.pt"
        )

        try:
            for epoch in range(1, max_epochs + 1):

                train_metrics = training.train_one_epoch(
                    model_run, train_loader,
                    optimizer_run, criterion_run, device
                )

                val_metrics = training.evaluate(
                    model_run, val_loader,
                    criterion_run, device
                )

                run.log({
                    "epoch": epoch,
                    "train/loss": train_metrics["loss"],
                    "train/accuracy": train_metrics["accuracy"],
                    "val/loss": val_metrics["loss"],
                    "val/accuracy": val_metrics["accuracy"]
                })

                improved = (
                    val_metrics["accuracy"] > best_acc
                    or (
                        val_metrics["accuracy"] == best_acc
                        and val_metrics["loss"] < best_loss
                    )
                )

                if improved:
                    best_acc = val_metrics["accuracy"]
                    best_loss = val_metrics["loss"]
                    best_epoch = epoch
                    no_improvement = 0

                    torch.save(
                        model_run.state_dict(), checkpoint
                    )
                else:
                    no_improvement += 1

                if no_improvement >= patience:
                    break

            # Verify saved checkpoint
            model_run.load_state_dict(
                torch.load(
                    checkpoint,
                    map_location=device,
                    weights_only=True
                )
            )

            restored = training.evaluate(
                model_run, val_loader,
                criterion_run, device
            )

            assert abs(
                restored["accuracy"] - best_acc
            ) < 1e-6

            run.summary["best_val_accuracy"] = best_acc
            run.summary["best_epoch"] = best_epoch
            run.summary["val_loss_at_best_accuracy"] = best_loss
            run.summary["epochs_trained"] = epoch

            multiseed_results.append({
                "seed": seed,
                "dropout": dropout,
                "best_accuracy": best_acc,
                "best_loss": best_loss,
                "best_epoch": best_epoch
            })

            print(
                f"Seed {seed} | Dropout {dropout} | "
                f"Accuracy {best_acc:.2%} | "
                f"Epoch {best_epoch}"
            )

        finally:
            run.finish()

# Aggregate results
print("\n=== MULTI-SEED COMPARISON ===")

for dropout in dropouts:
    scores = [
        r["best_accuracy"]
        for r in multiseed_results
        if r["dropout"] == dropout
    ]

    print(
        f"Dropout {dropout}: "
        f"Mean = {statistics.mean(scores):.2%}, "
        f"Std = {statistics.stdev(scores):.2%}, "
        f"Min = {min(scores):.2%}, "
        f"Max = {max(scores):.2%}"
    )

print("\n=== PAIRED SEED DIFFERENCES ===")

for seed in seeds:
    scores = {
        r["dropout"]: r["best_accuracy"]
        for r in multiseed_results
        if r["seed"] == seed
    }

    difference = 100 * (scores[0.2] - scores[0.0])

    print(
        f"Seed {seed}: "
        f"Dropout improvement = "
        f"{difference:+.2f} percentage points"
    )


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Seed 42 | Dropout 0.0 | Accuracy 63.10% | Epoch 41


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/accuracy,▁▂▃▃▄▄▄▄▄▄▅▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████
train/loss,██▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁
val/accuracy,▁▃▅▅▆▅▆▆▇▆▆▆▆▇▆▆▅▇█▇▇▇█▇███▇▇▇▇▅▇▇▇▇▇▇▇▆
val/loss,▆▃▃▂▂▂▁▂▂▁▂▂▁▂▄▂▃▅▂▂▂▃▃▃▂▃▅▃▃▅▄▅▄▄▆█▆▇▇▇
best_epoch,41
best_val_accuracy,0.631
epoch,61
epochs_trained,61
train/accuracy,0.83251
train/loss,0.40244


Seed 42 | Dropout 0.2 | Accuracy 65.00% | Epoch 69


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/accuracy,▁▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇████
train/loss,███▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▁▁▁
val/accuracy,▁▂▄▄▄▃▄▄▆▄▃▅▆▅▆▆▇▇▅▄▅▄█▅▅▅█▆▆▅█▇▆▆▆▆▆▄▆▇
val/loss,▄▃▃▂▃▂▁▂▂▂▁▁▁▁▂▂▂▂▂▂▃▃▂▃▃▃▄▄▄▅▄▅▅▅▇▆▆▇▇█
best_epoch,69
best_val_accuracy,0.65
epoch,89
epochs_trained,89
train/accuracy,0.88935
train/loss,0.29363


Seed 123 | Dropout 0.0 | Accuracy 62.90% | Epoch 28


... (4 outputs left out)

epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train/accuracy,▁▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████
train/loss,██▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁
val/accuracy,▁▃▄▄▅▅▆▆▆▅▆▆▅▆▆▆▅▅▆▇▆▇▄▆▇█▇▄▇▇▆▇▇█▇▆▅▇▆▆
val/loss,▃▃▂▂▁▂▁▁▂▁▁▂▁▂▁▂▂▂▂▁▆▃▂▂▃▂▃▂▃▇▃▄▆▄▄▅▄▆▇█
best_epoch,43
best_val_accuracy,0.639
epoch,63
epochs_trained,63
train/accuracy,0.82043
train/loss,0.41629


Seed 2026 | Dropout 0.2 | Accuracy 63.70% | Epoch 32


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/accuracy,▁▂▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████
train/loss,██▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁
val/accuracy,▁▄▄▅▆▆▆▆▆▆▆▆▆▇▆▇▇▇▆▇▆▇▇█▇█▇▇▇▇▇█▇▇▇█▇▇█▇
val/loss,█▆▅▄▇▃▃▅▂▂▂▂▂▂▄▁▂▂▁▁▂▁▁▃▂▁▂▂▂▄▅▅▂▂▄▃▅▃▅▅
best_epoch,32
best_val_accuracy,0.637
epoch,52
epochs_trained,52
train/accuracy,0.7984
train/loss,0.44271



=== MULTI-SEED COMPARISON ===
Dropout 0.0: Mean = 63.30%, Std = 0.53%, Min = 62.90%, Max = 63.90%
Dropout 0.2: Mean = 64.53%, Std = 0.72%, Min = 63.70%, Max = 65.00%

=== PAIRED SEED DIFFERENCES ===
Seed 42: Dropout improvement = +1.90 percentage points
Seed 123: Dropout improvement = +2.00 percentage points
Seed 2026: Dropout improvement = -0.20 percentage points


In [7]:
import wandb
from pathlib import Path

# Directory for checkpoints
checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Descriptive experiment name
run_name = (
    f"mlp_h{config['hidden_dim']}"
    f"_lr{config['learning_rate']}"
    f"_bs{config['batch_size']}"
    f"_seed{config['seed']}"
    "_preprocV2_60ep_ES"
)

# Initialize W&B
run = wandb.init(
    entity="kimbortis-hochschule-luzern",
    project="nlp-art-hs26-project1",
    group="fasttext-mlp",
    job_type="train",
    name=run_name,
    config={
        **config,
        "preprocessing": "v2",
        "vocab_size": len(vocab),
        "embedding_frozen": True
    }
)

# Plot all metrics against the epoch
run.define_metric("epoch")
run.define_metric("train/*", step_metric="epoch")
run.define_metric("val/*", step_metric="epoch")

# Unique checkpoint for this experiment
checkpoint_path = checkpoint_dir / (
    f"{run_name}_{run.id}_best.pt"
)

best_val_accuracy = -1.0
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

lowest_val_loss = float("inf")
lowest_loss_epoch = 0

print("W&B run:", run.name)
print("W&B URL:", run.url)
print("Checkpoint:", checkpoint_path)

try:
    for epoch in range(1, config["max_epochs"] + 1):

        # Training
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        # Validation
        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        # Log metrics against epoch
        run.log({
            "epoch": epoch,
            "train/loss": train_metrics["loss"],
            "train/accuracy": train_metrics["accuracy"],
            "val/loss": val_metrics["loss"],
            "val/accuracy": val_metrics["accuracy"]
        })

        # Track lowest validation loss independently
        if val_metrics["loss"] < lowest_val_loss:
            lowest_val_loss = val_metrics["loss"]
            lowest_loss_epoch = epoch

            run.summary["lowest_val_loss"] = lowest_val_loss
            run.summary["lowest_loss_epoch"] = lowest_loss_epoch

        print(
            f"Epoch {epoch:02d} | "
            f"Train loss: {train_metrics['loss']:.4f} | "
            f"Train acc: {train_metrics['accuracy']:.2%} | "
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val acc: {val_metrics['accuracy']:.2%}"
        )

        # Select by validation accuracy
        # Break ties using validation loss
        improved = (
            val_metrics["accuracy"] > best_val_accuracy
            or (
                val_metrics["accuracy"] == best_val_accuracy
                and val_metrics["loss"] < best_val_loss - 1e-4
            )
        )

        if improved:
            best_val_accuracy = val_metrics["accuracy"]
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            epochs_without_improvement = 0

            # Save model parameters
            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            # Store best results in W&B summary
            run.summary["best_epoch"] = best_epoch
            run.summary["best_val_accuracy"] = best_val_accuracy
            run.summary["val_loss_at_best_accuracy"] = best_val_loss

        else:
            epochs_without_improvement += 1

        # Early stopping (disabled for diagnostic runs)
        if (
            config["early_stopping"]
            and epochs_without_improvement >= config["patience"]
        ):
            print(f"\nEarly stopping at epoch {epoch}")
            break

    print("\n=== BEST MODEL ===")
    print("Best epoch:", best_epoch)
    print(f"Best validation accuracy: {best_val_accuracy:.2%}")
    print(f"Validation loss: {best_val_loss:.4f}")
    print("\n=== LOWEST VALIDATION LOSS ===")
    print("Epoch:", lowest_loss_epoch)
    print(f"Lowest validation loss: {lowest_val_loss:.4f}")
    print("Checkpoint:", checkpoint_path)

finally:
    run.finish()

NameError: name 'config' is not defined

In [ ]:
# Restore the best model checkpoint
model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

# Evaluate the restored model
best_metrics = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("=== RESTORED BEST MODEL ===")
print("Best epoch:", best_epoch)
print(f"Validation loss: {best_metrics['loss']:.4f}")
print(f"Validation accuracy: {best_metrics['accuracy']:.2%}")

# Verify checkpoint matches the recorded best result
assert abs(
    best_metrics["accuracy"] - best_val_accuracy
) < 1e-6

print("Checkpoint verification: PASS")

# Compare against the previous baseline
previous_accuracy = 0.622

difference = (
    best_metrics["accuracy"] - previous_accuracy
) * 100

print("\n=== BASELINE COMPARISON ===")
print(f"Previous MLP: {previous_accuracy:.2%}")
print(f"Updated MLP: {best_metrics['accuracy']:.2%}")
print(f"Difference: {difference:+.2f} percentage points")
